### Date: 7/30/2025

This will serve as a guide for running QC and alignment for the ATAC-seq using nf-core/atacseq. The nf-core pipeline is only used to generate aligned reads (using bwa). These bams are then filtered to only contain autosomes and merged for peak calling separately with MACS3

Sample information is provided via a csv file. The pipeline will auto-detect whether a sample is single- or paired-end using the information provided in the samplesheet. The samplesheet can have as many columns as you desire, however, there is a strict requirement for the first 4 columns to be the following:

* sample: sample name
* fastq_1: path to the first fastq file
* fastq_2: path to the second fastq file
* replicate: replicate number

In [2]:
# generate a sample list to be used by the nextflow pipeline

import pandas as pd

sample_list = pd.read_csv("/gs/gsfs0/shared-lab/greally-lab/David/chromBPnet_analysis/nf-core_atac/sample_list.csv")

# print the sample list
print(sample_list)

              sample                                            fastq_1  \
0       neutrophil_1  /gs/gsfs0/shared-lab/greally-lab/David/chromBP...   
1       neutrophil_1  /gs/gsfs0/shared-lab/greally-lab/David/chromBP...   
2       neutrophil_2  /gs/gsfs0/shared-lab/greally-lab/David/chromBP...   
3       neutrophil_2  /gs/gsfs0/shared-lab/greally-lab/David/chromBP...   
4       neutrophil_3  /gs/gsfs0/shared-lab/greally-lab/David/chromBP...   
5       neutrophil_3  /gs/gsfs0/shared-lab/greally-lab/David/chromBP...   
6         cd4_t_cell  /gs/gsfs0/shared-lab/greally-lab/David/chromBP...   
7   cd19_cd20_b_cell  /gs/gsfs0/shared-lab/greally-lab/David/chromBP...   
8         cd8_t_cell  /gs/gsfs0/shared-lab/greally-lab/David/chromBP...   
9            nk_cell  /gs/gsfs0/shared-lab/greally-lab/David/chromBP...   
10     cd14_monocyte  /gs/gsfs0/shared-lab/greally-lab/David/chromBP...   
11            pbmc_1  /gs/gsfs0/shared-lab/greally-lab/David/chromBP...   
12            pbmc_2  /gs

In [1]:
import os
bash = "/gs/gsfs0/shared-lab/greally-lab/David/chromBPnet_analysis/nf-core_atac/bash"

!sbatch $bash/run_nf-core_atac.sh

Submitted batch job 14426373


In [ ]:
# Now, using the aligned bams, generate a filtered set of bams that only contain reads mapped to autosomes. The filtered bms will be used for peak calling, for training the chromBPnet Tn5 bias model. 
# Replicates will be merged into a single bam file for each sample.
# This script will also generate a read count summary for each bam file.

bash = "/gs/gsfs0/shared-lab/greally-lab/David/chromBPnet_analysis/nf-core_atac/bash"
input_dir = "/gs/gsfs0/shared-lab/greally-lab/David/chromBPnet_analysis/nf-core_atac/results/bwa/merged_library"
output_dir = "/gs/gsfs0/shared-lab/greally-lab/David/chromBPnet_analysis/nf-core_atac/results/bwa/merged_library/autosomes_only"

!sbatch $bash/filter_bwa_bams.sh $input_dir $output_dir


Submitted batch job 14090958


In [ ]:
# Now, using the aligned bams, generate a filtered set of bams that only contain reads mapped to autosomes. The filtered bms will be used for peak calling, for training the chromBPnet Tn5 bias model. 
# Run for the merged pbmc samples (pbmc-2_4_merged)

bash = "/gs/gsfs0/shared-lab/greally-lab/David/chromBPnet_analysis/nf-core_atac/bash"
input_dir = "/gs/gsfs0/shared-lab/greally-lab/David/chromBPnet_analysis/nf-core_atac/results/bwa/merged_library/merged_pbmcs_2-4"
output_dir = "/gs/gsfs0/shared-lab/greally-lab/David/chromBPnet_analysis/nf-core_atac/results/bwa/merged_library/autosomes_only/merged_pbmcs_2-4"

!sbatch $bash/filter_bwa_bams.sh $input_dir $output_dir

# merged set has 195,564,296 aligned (hg38) reads. (date: 8/19/25)




Submitted batch job 14209898


### Date:9/5/2025

Going to re-run chrombpnet using a larger merged PBMC set (8 samples).
All samples listed in : sample_list_pbmc.csv




In [ ]:
import os
bash = "/gs/gsfs0/shared-lab/greally-lab/David/chromBPnet_analysis/nf-core_atac/bash"

!sbatch $bash/run_nf-core_atac.sh

In [9]:
# Now, using the aligned bams, generate a filtered set of bams that only contain reads mapped to autosomes. The filtered bms will be used for peak calling, for training the chromBPnet Tn5 bias model. 
# Run for the merged pbmc samples
import os


bash = "/gs/gsfs0/shared-lab/greally-lab/David/chromBPnet_analysis/nf-core_atac/bash"
input_dir = "/gs/gsfs0/shared-lab/greally-lab/David/chromBPnet_analysis/nf-core_atac/results_9-5-2025/bwa/merged_library"
output_dir = "/gs/gsfs0/shared-lab/greally-lab/David/chromBPnet_analysis/nf-core_atac/results_9-5-2025/bwa/merged_library/autosomes_only/pbmcs_2-9-parallel"

!sbatch --array=1-8 $bash/filter_bwa_bams_parallel.sh $input_dir $output_dir 





Submitted batch job 14577613


In [2]:
# Merge the pbmc samples into a single bam file. 

bash = "/gs/gsfs0/shared-lab/greally-lab/David/chromBPnet_analysis/nf-core_atac/bash"
input_dir = "/gs/gsfs0/shared-lab/greally-lab/David/chromBPnet_analysis/nf-core_atac/results_9-5-2025/bwa/merged_library/autosomes_only/pbmcs_2-9-parallel"
output_dir = "/gs/gsfs0/shared-lab/greally-lab/David/chromBPnet_analysis/nf-core_atac/results_9-5-2025/bwa/merged_library/autosomes_only/pbmcs_2-9-parallel/merged"

!sbatch $bash/merge_bams.sh $input_dir $output_dir

Submitted batch job 14581977


### DATE: 11-3-25
Run for new neutrophil dataset



In [1]:
# Configure params in params.yaml and bash/run_nf-core_atac.sh (they should match)
import os
bash = "/gs/gsfs0/shared-lab/greally-lab/David/chromBPnet_analysis/nf-core_atac/bash"
input = "/gs/gsfs0/shared-lab/greally-lab/David/chromBPnet_analysis/nf-core_atac/sample_list_neutrophil_set-1.csv"
outdir = "/gs/gsfs0/shared-lab/greally-lab/David/chromBPnet_analysis/nf-core_atac/results/11-11-25/neutrophil_set-1" 
!sbatch $bash/run_nf-core_atac.sh $input $outdir 

# input = "/gs/gsfs0/shared-lab/greally-lab/David/chromBPnet_analysis/nf-core_atac/sample_list_neutrophil_set-2.csv"
# outdir = "/gs/gsfs0/shared-lab/greally-lab/David/chromBPnet_analysis/nf-core_atac/results/11-11-25/neutrophil_set-2" 
# !sbatch $bash/run_nf-core_atac.sh $input $outdir

Submitted batch job 22555069


In [ ]:
# Now, using the aligned bams, generate a filtered set of bams that only contain reads mapped to autosomes. The filtered bams will be merged into 1 file and used for peak calling, for training the chromBPnet Tn5 bias model. 
import os


bash = "/gs/gsfs0/shared-lab/greally-lab/David/chromBPnet_analysis/nf-core_atac/bash"
input_dir = "/gs/gsfs0/shared-lab/greally-lab/David/chromBPnet_analysis/nf-core_atac/results/bwa/merged_library/neutrophil"
output_dir = "/gs/gsfs0/shared-lab/greally-lab/David/chromBPnet_analysis/nf-core_atac/results/bwa/merged_library/autosomes_only/neutrophil" 

!sbatch --array=1-3 $bash/filter_bwa_bams_parallel.sh $input_dir $output_dir  





In [1]:
# Merge the neutrophil samples into a single bam file. Only merges the 2 high quality samples. 

bash = "/gs/gsfs0/shared-lab/greally-lab/David/chromBPnet_analysis/nf-core_atac/bash"
input_dir = "/gs/gsfs0/shared-lab/greally-lab/David/chromBPnet_analysis/nf-core_atac/results/bwa/merged_library/autosomes_only/neutrophil" 
output_dir = "/gs/gsfs0/shared-lab/greally-lab/David/chromBPnet_analysis/nf-core_atac/results/bwa/merged_library/autosomes_only/neutrophil/merged" 

!sbatch $bash/merge_bams.sh $input_dir $output_dir

Submitted batch job 22475692


### DATE: Re-run both neutrophil sets using defualt params.



In [ ]:
# Configure params in params.yaml and bash/run_nf-core_atac.sh (they should match)
import os
# bash = "/gs/gsfs0/shared-lab/greally-lab/David/chromBPnet_analysis/nf-core_atac/bash"
# input = "/gs/gsfs0/shared-lab/greally-lab/David/chromBPnet_analysis/nf-core_atac/sample_list_neutrophil_set-1.csv"
# outdir = "/gs/gsfs0/shared-lab/greally-lab/David/chromBPnet_analysis/nf-core_atac/results/11-11-25/neutrophil_set-1" 
# !sbatch $bash/run_nf-core_atac.sh $input $outdir 

input = "/gs/gsfs0/shared-lab/greally-lab/David/chromBPnet_analysis/nf-core_atac/sample_list_neutrophil_set-2.csv"
outdir = "/gs/gsfs0/shared-lab/greally-lab/David/chromBPnet_analysis/nf-core_atac/results/11-11-25/neutrophil_set-2" 
!sbatch $bash/run_nf-core_atac.sh $input $outdir

Submitted batch job 22555433
